# Cell 1 — Load Data

In [ ]:
import pandas as pd
import numpy as np
from joblib import load

# Load your trained stacked model
AE_IF = load("model/AE_IF_stacking.joblib")

# Load new flow-level feature data
df = pd.read_csv("data/cleaned_KPIs.csv")
# Extract features (same features used during training)
feature_cols = AE_IF["features"]       
X = df[feature_cols].values

# Cell 2 — Generate Anomaly Score from Model

In [ ]:
# The model must output anomaly score
scores = AE_IF["model"].decision_function(X)   # <--- If using IsolationForest in stack

df["anomaly_score"] = scores

# If labels exist (optional, for evaluation):
if "Label" in df.columns:
    df["y_true"] = df["Label"].astype(str).str.lower().isin(["benign","normal"]).map({True:0, False:1})

# Cell 3 — Double Threshold Binary Classification

T1 = 0.35
T2 = 0.45

# Primary confident decisions:
df["primary_pred"] = np.where(df["anomaly_score"] >= T2, 1, 0)

# Identify uncertain region
df["uncertain"] = (df["anomaly_score"] >= T1) & (df["anomaly_score"] < T2)

print("Confident Normal:", (df["anomaly_score"] < T1).sum())
print("Uncertain Zone:", df["uncertain"].sum())
print("Confident Anomaly:", (df["anomaly_score"] >= T2).sum())

# Cell 4 — Secondary Verification on Uncertain Zone

In [ ]:
from sklearn.ensemble import IsolationForest

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove("anomaly_score")

# train verifier only on confident-normal region:
X_train = df.loc[df["anomaly_score"] < T1, numeric_cols].values

verifier = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42
)
verifier.fit(X_train)

# Apply to uncertain rows:
idx_uncertain = df.index[df["uncertain"]]
X_uncertain = df.loc[idx_uncertain, numeric_cols].values

sec_pred = verifier.predict(X_uncertain)  # 1 = normal, -1 = anomaly

# Map to binary: -1 → anomaly, 1 → normal
df.loc[idx_uncertain, "secondary_pred"] = (sec_pred == -1).astype(int)

# Final decision:
df["y_pred_bin"] = df["primary_pred"]
df.loc[idx_uncertain, "y_pred_bin"] = df.loc[idx_uncertain, "secondary_pred"]


# EVALUATION

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

acc  = accuracy_score(df["y_true"], df["y_pred_bin"])
prec = precision_score(df["y_true"], df["y_pred_bin"])
rec  = recall_score(df["y_true"], df["y_pred_bin"])
f1   = f1_score(df["y_true"], df["y_pred_bin"])
auc  = roc_auc_score(df["y_true"], df["anomaly_score"])

cm = confusion_matrix(df["y_true"], df["y_pred_bin"])

print(" FINAL EVALUATION (Double Threshold + Verification)")
print(f"Accuracy   : {acc:.4f}")
print(f"Precision  : {prec:.4f}")
print(f"Recall     : {rec:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"ROC-AUC    : {auc:.4f}")
print("\nConfusion Matrix (TN, FP / FN, TP):")
print(cm)

print("\nDetailed Report:")
print(classification_report(df["y_true"], df["y_pred_bin"], digits=4))


FINAL EVALUATION (Double Threshold + Verification)
Accuracy   : 0.9482
Precision  : 0.9047
Recall     : 0.8294
F1 Score   : 0.8655
ROC-AUC    : 0.9651

Confusion Matrix (TN, FP / FN, TP):
[[65241   812]
 [ 1517  7349]]

Detailed Report:
              precision    recall  f1-score   support
Normal         0.97      0.99      0.98      66053
Attack         0.90      0.83      0.86       8866
